In [10]:
import numpy as np
import pandas as pd
import yaml
import os
from matplotlib import pyplot as plt
from hsa_hopper.kinematics import *
from hsa_hopper.controller import HopController
from hsa_hopper.hsa_model import HSAModel
actuator_model_path = r'/home/joseph/workspaces/hsa_hopper/hsa_hopper_control/notebooks/actuator_model.yaml'
with open(actuator_model_path, 'r') as f:
    actuator_model = yaml.load(f, yaml.Loader)

char_model_path = r'/home/joseph/workspaces/hsa_hopper/hsa_hopper_control/notebooks/aim2/nlsd_model.yaml'
with open(char_model_path, 'r') as f:
    hsa_char_model = HSAModel.make_from_dict(yaml.load(f, yaml.Loader))
kin_params = KinematicParameters(.07,.15,.3,-.005)

def load_data(root, include_dynamics_params=True):
    dataframes = {}
    hardware_configs = {}
    dynamics_params = {}
    controller_params = {}

    for category in os.listdir(root):
        print(category)
        dataframes[category] = {}
        hardware_configs[category] = {}
        dynamics_params[category] = {}
        controller_params[category] = {}
        category_folder = os.path.join(root,category)
        for trial in os.listdir(category_folder):
            trial_folder = os.path.join(category_folder, trial)
            trial_df = pd.read_csv(os.path.join(trial_folder,'data.csv'))
            dataframes[category][trial] = trial_df
            # load config files
            with open(os.path.join(trial_folder,'hardware_config.yaml')) as f:
                hardware_configs[category][trial] = yaml.load(f,yaml.Loader)
            with open(os.path.join(trial_folder,'experiment_config.yaml')) as f:
                experiment_config = yaml.load(f,yaml.Loader)
                servo_pos = experiment_config['controller']['servo_pos']
                trial_df['psi'] = (servo_pos-1500)*(np.pi/4)*200
                if include_dynamics_params:
                    dynamics_params[category][trial] = experiment_config['dynamics_params']
                    m_foot = dynamics_params[category][trial]['m_foot']
                    m_cart = dynamics_params[category][trial]['m_cart'] 
                    total_mass = m_cart+m_foot
                    trial_df['total_mass'] = m_cart+m_foot 
                else:
                    total_mass = int(category.split('_')[-1][:-1])/1000
                    m_foot = actuator_model['m_foot']
                    trial_df['total_mass'] = total_mass
                    m_cart = total_mass - m_foot
                controller_params[category][trial] = experiment_config['controller']
            # load (smoothed) data files
    return dataframes, hardware_configs, dynamics_params, controller_params


In [11]:
# aim1_path = r'/home/joseph/workspaces/hsa_hopper/hsa_hopper_control/data/hop_experiments/design_paper'
# aim1_data = load_data(aim1_path, include_dynamics_params = False)
aim2_path = r'/home/joseph/workspaces/hsa_hopper/hsa_hopper_control/data/hopping_data'
aim2_data = load_data(aim2_path, include_dynamics_params = True)

no_hsa_1500g
hsa_1700g
no_hsa_1900g
hsa_1500g
no_hsa_2100g
no_hsa_1300g
hsa_1900g
hsa_1300g
no_hsa_1700g
hsa_2100g


In [12]:
R = 1.5*.095
gear_ratio = 6
Km = (105) # 105 RPM per Volt
Kt = gear_ratio*(3/2)/np.sqrt(3)*(60/(2*np.pi))/Km # torque constant in Nm/A or Volts * seconds
print(Kt)
# run these on each trials dataframe individually - do not concatenate separate trials

# do power calculations and add them to the data frame
def power_calcs(df: pd.DataFrame):
    x = df['x_rad'].to_numpy()
    xdot = df['xdot'].to_numpy()
    xddot = df['xddot'].to_numpy()
    ydot = df['ydot'].to_numpy()
    yddot = df['yddot'].to_numpy()
    tau = df['torque'].to_numpy()
    total_mass = df['total_mass'].to_numpy()
    mode = df['mode'].to_numpy()
    m_cart = total_mass - actuator_model['m_foot']
    unique_mode = df['mode'].unique()
    if len(unique_mode) == 3:
        # this data is from the design paper
        leg_inertia = ((mode==2)+(mode==3))*m_cart+(mode==1)*actuator_model['m_foot']
    else:
        # this data is from the modeling paper
        leg_inertia = (mode == HopController._STANCE)*m_cart+(mode == HopController._FLIGHT)*actuator_model['m_foot']
    therm_power = R*(tau/Kt)**2
    joint_power = actuator_model['J']*xddot*xdot + leg_inertia*(yddot+9.81)*ydot
    motor_power = tau*xdot
    elec_power = therm_power + motor_power
    df['therm_power'] = therm_power
    df['joint_power'] = joint_power
    df['motor_power'] = motor_power
    df['elec_power'] = elec_power

def energy_calcs(df: pd.DataFrame):
    hop_idx = np.sort(df['hop_idx'].unique())
    therm_work = np.zeros(len(hop_idx)-1)
    joint_work = np.zeros_like(therm_work)
    motor_work = np.zeros_like(therm_work)
    elec_work = np.zeros_like(therm_work)
    hop_height = np.zeros_like(therm_work)
    COT = np.zeros_like(therm_work)
    TCOT = np.zeros_like(therm_work)
    JCOT = np.zeros_like(therm_work)
    MCOT = np.zeros_like(therm_work)
    unique_modes = df['mode'].unique()
    for i, idx in enumerate(hop_idx[:-1]):
        frame = df['hop_idx'] == idx
        t = df.loc[frame,'t_s'].to_numpy()
        P = df.loc[frame,'therm_power'].to_numpy()
        therm_work[i] = np.trapz(P,t)
        P = df.loc[frame,'joint_power'].to_numpy()
        joint_work[i] = np.trapz(P,t)
        P = df.loc[frame,'motor_power'].to_numpy()
        motor_work[i] = np.trapz(P,t)
        P = df.loc[frame,'elec_power'].to_numpy()
        elec_work[i] = np.trapz(P,t)
        y = df.loc[frame, 'y_m'].to_numpy()
        # x_raw = df.loc[frame, 'x_raw'].to_numpy()
        # fk,jac = forward_kinematics(kin_params, x_raw, jacobian=True)
        ydot = df.loc[frame, 'ydot'].to_numpy()
        t = df.loc[frame, 't_s'].to_numpy()
        mode = df.loc[frame, 'mode'].to_numpy()
        # if len(unique_modes) == 3:
        #     # this data is from the design paper
        #     flight_start = next(i for i in range(len(mode)) if mode[i] == 1)
        # else:
        flight_start = next(i for i in range(len(mode)) if mode[i] == HopController._FLIGHT)
            # this data is from the modeling paper
        # xdot = (x_raw[flight_start]-x_raw[flight_start-1])/(t[flight_start]-t[flight_start-1])
        # ydot = (y[flight_start-1]-y[flight_start-2])/(t[flight_start-1]-t[flight_start-2])
        # ydot = jac[0]*xdot
        # hop_height[i] = (ydot**2)/(2*9.81)
        hop_height[i] = (ydot[flight_start-1]**2)/(2*9.81)
        total_mass = df.loc[frame,'total_mass'].to_numpy()[0]
        cost_norm = total_mass*9.81*hop_height[i]
        COT[i] = elec_work[i]/cost_norm
        TCOT[i] = therm_work[i]/cost_norm
        JCOT[i] = joint_work[i]/cost_norm
        MCOT[i] = motor_work[i]/cost_norm
    return pd.DataFrame({
        'hop_idx': hop_idx[:-1],
        'therm_work': therm_work,
        'joint_work' : joint_work,
        'motor_work' : motor_work,
        'elec_work' : elec_work,
        'hop_height' : hop_height,
        'COT' : COT,
        'TCOT' : TCOT,
        'JCOT' : JCOT,
        'MCOT' : MCOT,
    })

# aim1_energy = {}
# for condition, trials in aim1_data[0].items():
#     aim1_energy[condition] = {}
#     for trial, df in trials.items():
#         power_calcs(df)
#         aim1_energy[condition][trial] = energy_calcs(df)
aim2_energy = {}
for condition, trials in aim2_data[0].items():
    aim2_energy[condition] = {}
    for trial, df in trials.items():
        power_calcs(df)
        aim2_energy[condition][trial] = energy_calcs(df)

0.47256762464725044


In [13]:
for condition, trials in aim1_energy.items():
    for trial, result in trials.items():
        print(condition)
        hop_height_mean = result['hop_height'].mean()
        cot_mean = result['COT'].mean()
        print(f'average hop height: {hop_height_mean}')
        print(f'average COT: {cot_mean}')

NameError: name 'aim1_energy' is not defined

In [14]:
for condition, trials in aim2_energy.items():
    for trial, result in trials.items():
        print(condition)
        print(aim2_data[3][condition][trial]['servo_pos'])
        hop_height_mean = result['hop_height'].mean()
        cot_mean = result['COT'].mean()
        print(f'average hop height: {hop_height_mean}')
        print(f'average COT: {cot_mean}')

no_hsa_1500g
1500
average hop height: 0.047355934540953576
average COT: 2.4120141439133995
hsa_1700g
1500
average hop height: 0.042384863534601615
average COT: 2.1392042755003566
hsa_1700g
1600
average hop height: 0.04014937554571668
average COT: 2.053559736149787
hsa_1700g
1700
average hop height: 0.042015711224580685
average COT: 1.9799010546988953
hsa_1700g
1400
average hop height: 0.04376600665452447
average COT: 2.115680229989176
no_hsa_1900g
1500
average hop height: 0.047690324301319115
average COT: 2.5180934712799714
hsa_1500g
1500
average hop height: 0.04148913062534915
average COT: 2.1650578985780355
hsa_1500g
1700
average hop height: 0.04315271236413915
average COT: 2.0259470218738658
hsa_1500g
1400
average hop height: 0.040612204800659114
average COT: 2.1017788679418574
hsa_1500g
1600
average hop height: 0.03940916555479037
average COT: 2.0770482195672924
no_hsa_2100g
1500
average hop height: 0.049118234530729596
average COT: 2.4973253617473508
no_hsa_1300g
1500
average hop 